In [1]:
%pip install kafka kafka-python-ng


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
kafka_url = 'localhost:9092'

In [7]:
import time
from kafka import KafkaProducer

kafka_url = 'localhost:9092'

# Démarrer le producteur Kafka
producer = KafkaProducer(
    bootstrap_servers=kafka_url, value_serializer=lambda v: str(v).encode('utf-8')
)

In [8]:
topic = 'topic-example'

for i in range(10):
    message = f'Message {i}'
    producer.send(topic, value=message)
    print(f'Envoyé: {message}')
    time.sleep(1)  # Pause 1sec pour simuler un flux

producer.flush()
producer.close()

Envoyé: Message 0
Envoyé: Message 1
Envoyé: Message 2
Envoyé: Message 3
Envoyé: Message 4
Envoyé: Message 5
Envoyé: Message 6
Envoyé: Message 7
Envoyé: Message 8
Envoyé: Message 9


In [3]:
from kafka import KafkaConsumer

topic = 'topic-example'
consumer = KafkaConsumer(
    topic,
    bootstrap_servers=[kafka_url],
    auto_offset_reset='earliest',
    consumer_timeout_ms=1000,
    value_deserializer=lambda value: value.decode('utf-8'),
)

print("Liste des messages reçus:")
for message in consumer:
    print(f'Reçu: {message.value}')

consumer.close()

Liste des messages reçus:
Reçu: Message 0
Reçu: Message 1
Reçu: Message 2
Reçu: Message 3
Reçu: Message 4
Reçu: Message 5
Reçu: Message 6
Reçu: Message 7
Reçu: Message 8
Reçu: Message 9


# Administration :

In [10]:
from kafka.admin import KafkaAdminClient, NewTopic

admin_client = KafkaAdminClient(
    bootstrap_servers=[kafka_url],
    client_id="admin",
)

topics = admin_client.list_topics()

print(topics)

['topic-example']


In [11]:
new_topic = 'my-custom-topic'

if new_topic not in topics:
    topic = NewTopic(
        name=new_topic,
        num_partitions=1,
        replication_factor=1
    )
    admin_client.create_topics([topic])

In [12]:
topics = admin_client.list_topics()

print(topics)

['my-custom-topic', 'topic-example']


In [14]:
admin_client.delete_topics([new_topic])

DeleteTopicsResponse_v3(throttle_time_ms=0, topic_error_codes=[(topic='my-custom-topic', error_code=0)])

In [15]:
topics = admin_client.list_topics()

print(topics)

['topic-example']


# Supprimer/Reset toutes les données d'un topic

In [22]:
from kafka.admin import ConfigResource, ConfigResourceType
from time import sleep

resource = ConfigResource(
    ConfigResourceType.TOPIC,
    topic,
    # Je lui dis de passer la rétention des données à 0ms
    # Donc ici il ne conservera les données reçue que 0ms (donc jamais)
    configs={ 'retention.ms': '0' }
)

# Appliquer la modification de configuration
admin_client.alter_configs([resource])
sleep(10)

# !! Après il important de remettre les valeurs par défaut :
resource = ConfigResource(
    ConfigResourceType.TOPIC,
    topic,
    # Je repasse la rétention des données à 7 jours
    configs={ 'retention.ms': str(7 * 24 * 3600 * 1000) }
)
admin_client.alter_configs([resource])

AlterConfigsResponse_v1(throttle_time_ms=0, resources=[(error_code=0, error_message=None, resource_type=2, resource_name='topic-example')])

In [ ]:
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=kafka_url,
    value_serializer=lambda v: str(v).encode('utf-8')
)

person_topic='persons'
with open('data/test.csv') as file:
    lines = file.readlines()
    for line in lines:
        producer.send(person_topic, value=line.strip().encode('utf-8'))

    producer.flush()
    producer.close()

In [10]:
from kafka import KafkaConsumer

comsumer = KafkaConsumer(
    person_topic,
    bootstrap_servers=[kafka_url],
    consumer_timeout_ms=1000,
    # Si je veux consomer uniquement les dernières données
    # Je peux passer en 'latest' au lieu de 'earliest'
    auto_offset_reset='earliest',
    value_deserializer=lambda value: value.decode('utf-8'),
)

for message in comsumer:
    print(f'message : {message.value}')

consumer.close()

message : b'test;asdf'
message : b'test;qwer'
message : b'test;zxcv'
message : b'test;asdf'
message : b'test;qwer'
message : b'test;zxcv'


KeyboardInterrupt: 